# Tests of format modifications

In [ ]:
from gammapy.data import DataStore, Observations
from gammapy.utils.scripts import make_path
import numpy as np
path = make_path("$GAMMAPY_DATA/tests/format/swgo/")

## Minimal case
Minimal solution for swgo adding `EVENT_TYPE` to `obs-index` and `hdu-index`



In [ ]:
datastore = DataStore.from_dir(path, "hdu-index.fits.gz", "obs-index.fits.gz")


In [ ]:
print(datastore.hdu_table)

In [ ]:
print(datastore.obs_table)

In [ ]:
#this would work on gammapy main wihtout any mofifications
for event_type in np.unique(datastore.obs_table["EVENT_TYPE"]):
    obs_selection = datastore.obs_table["EVENT_TYPE"]==event_type
    hdu_selection = datastore.hdu_table["EVENT_TYPE"]==event_type
    datastore_redu = DataStore(hdu_table=datastore.hdu_table[hdu_selection],
                          obs_table=datastore.obs_table[obs_selection])
    observations = datastore_redu.get_observations()

If we want also multiple irf per observation, this won't work so we need to introduce a new keywords to idenfy uniques irfs and independant data partition :

- Each combination of event_type and observation_block (or any future independent data partition) should be associated with a fixed
`IRF_NAME`.

- A given `IRF_NAME` might not be associated with a well-defined IRF; in such cases, the `FILE_NAME` in the `hdu-index` would be empty.

- Any identifier used to select independent data partitions, such as `OBS_ID`, `EVENT_TYPE`, `OBS_BLOCK_ID`, and `IRF_NAME`, should only appear in the obs-index and not in the hdu-index (to avoid duplicate entries).

  
- A new unique identifier should be introduced on both obs-index and hdu-index to link each independent data entry in the obs-table to the corresponding files in the hdu-table. This could be `OBS_TABLE_ROW` (or maybe more generally `PARTITION_ID`).

- (Alternatively to deviate the least possible from GADF we can keep the `OBS_ID` in both obs-table the hdu-table but it should be then be interpreted as an unique independent data entry identifier equivalent to a `OBS_TABLE_ROW`. While the present `OBS_ID` will be introduced as a new identified `RUN_ID` and  `OBS_BLOCK_ID` would be instead `RUN_BLOCK_ID` or `CHUNK_ID`. Pesonnaly I prefer the solution given by the previous point).



In [ ]:
datastore = DataStore.from_dir(path, "hdu-index-irf-name.fits.gz", "obs-index-irf-name.fits.gz")


In [ ]:
datastore.hdu_table

In [ ]:
datastore.obs_table

In [ ]:
#this requires PR 5687

for irf_name in np.unique(datastore.obs_table["IRF_NAME"]): #for SWGO it could be simply EVENT_TYPE
    obs_selection = datastore.obs_table["IRF_NAME"]==irf_name
    obs_selection &= (datastore.obs_table["ALT_PNT"]>=0) & (datastore.obs_table["ALT_PNT"]<=90)
    hdu_selection = np.array([row in datastore.obs_table["OBS_TABLE_ROW"][obs_selection] for row in datastore.hdu_table["OBS_TABLE_ROW"]])
    datastore_redu = DataStore(hdu_table=datastore.hdu_table[hdu_selection],
                      obs_table=datastore.obs_table[obs_selection])

    observations = datastore_redu.get_observations()

print(list(observations))

I would remore obs_id argument from get_observations and do all selection with a new method `datastore.select` for example : 

In [ ]:
def datastore_select(datastore, selection_dict=None, selection_mask=None, strict=False,  unique_key=None):

    #the name of the unique identifier might also not be fixed 
    if unique_key is None:
        if "OBS_TABLE_ROW" in datastore.obs_table.keys():
            unique_key = "OBS_TABLE_ROW"
        elif "OBS_ID" in unique_key in datastore.obs_table.keys():
            unique_key = "OBS_ID"
    if len(datastore.obs_table[unique_key])!=len(np.unique(datastore.obs_table[unique_key])):
        raise KeyError(f" `unique_key` {unique_key} has non unique entries.")

    if selection_mask:
        obs_selection = selection_mask
    else:
        obs_selection = np.ones(len(datastore.obs_table), dtype=bool)

    if selection_dict:
        for key, value in selection_dict.items():
            if not strict and key not in datastore.obs_table.keys():
                #ignore and give some warning
                continue
            if isinstance(value, list):
                obs_selection &= np.array([ _ in value for _ in datastore.obs_table[key]])
            elif isinstance(value, range):
                obs_selection &= datastore.obs_table[key] >= value[0]
                obs_selection &= datastore.obs_table[key] <= value[-1]+ value[1]-value[0]
            else:
                obs_selection &= datastore.obs_table[key] == value
    hdu_selection = np.array([row in datastore.obs_table[unique_key][obs_selection] for row in datastore.hdu_table[unique_key]])
    datastore_redu = DataStore(hdu_table=datastore.hdu_table[hdu_selection],
                      obs_table=datastore.obs_table[obs_selection])
    return datastore_redu

selection_dict = {"EVENT_TYPE": ["Zen [45, 52] Cor [58, 82]", "Zen [45, 52] Cor [0, 58]" ], "ALT_PNT":range(0, 90), "OBS_ID":0}
datastore_redu = datastore_select(datastore, selection_dict)
observations = datastore_redu.get_observations()

print(list(observations))

The datastore cloud also implement a `get_observations_group` that re-use `observation.group_by_label`

In [ ]:
def datastore_get_observation_groups(datastore, key):
    observations = datastore.get_observations()
    observations_group = observations.group_by_label(datastore.obs_table[key])
    reformated = dict()
    for old_key, value in observations_group.items():
        reformated[f"{key}{old_key[5:]}"] = value
    return reformated 
    
observations_groups = datastore_get_observation_groups(datastore_redu, "IRF_NAME")
print(observations_groups)

If we have SWGO data slitted in chunks like HAWC (because all the events cannot fit in a single file) then we will have multiple entries with the same irfs. This would be equivalent to have multiple obsersvation or obsersvation block with the same irf.

In [ ]:
datastore = DataStore.from_dir(path, "hdu-index-multi.fits.gz", "obs-index-multi.fits.gz")


In [ ]:
datastore.obs_table

For CTA the observation blocks will be a succession of stable table interval, (STI associated to an irf), and unstable time interval (UTI not associated to an irf but still defined as a fixed data partition):

`observation |0                            |`

`blocks      |0   |1   |2   |3   |4   |5   |`

`irf         |null|a   |a   |b   |null|c   |`

later on we could even redefine the irfs of some blocks without acutally changing the partition of the events

`irf         |d   |a   |b   |b   |null|c   |`



In [ ]:
#this requires PR 5687

observations = list(datastore_get_observation_groups(datastore, "IRF_NAME").values())[0]
print("number of observations", len(observations))


Ideallly `get_observations` should stack the entries with the same `IRF_name` to avoid repeating data reduction 
and allows `obs.events` to be a list of events lists to avoid having everything in memory in that case

In [ ]:
#something like this 
def stack_observations(observations):
    obs = observations[0].copy()
    obs._events = (_._events for _ in observations)
    return obs

obs = stack_observations(observations)
print("obs_id", obs.obs_id)
print("events:", obs.events)
print("events lists:", list(obs.events))


Stacking same irfs together could done directly on `datastore.get_observations`


In [ ]:
def datastore_get_observations(datastore, stack_same_irfs=True):
    if not stack_same_irfs or not "IRF_NAME" in datastore.obs_table.keys():
        return datastore.get_observations()
    else:
        observation_groups = datastore_get_observation_groups(datastore, "IRF_NAME")
        return Observations([stack_observations(_) for _ in observation_groups.values()])

In [ ]:
observations = datastore_get_observations(datastore)
print("number of obs_table entries", len(datastore.obs_table))
print("number of observations", len(observations))